In [1]:
#Step 1
import pandas as pd
df_general = pd.read_csv('FlightInfo_general.csv', delimiter=';')
df_times = pd.read_csv('FlightInfo_times.csv')
df_general = df_general.drop_duplicates(subset=['FlightID'])
df = pd.merge(df_general,df_times,on='FlightID')

In [2]:
df['Scheduled_Time'] = (df['ScheduledDeptTime'].astype(str).str[:-2] + ':' + df['ScheduledDeptTime'].astype(str).str[:-2])
df['Actual_Time'] = (df['ActualDeptTime'].astype(str).str[:-2] + ':' + df['ActualDeptTime'].astype(str).str[:-2])

In [3]:
df['scheduled_full'] = pd.to_datetime(df['Date'] + ' ' + df['Scheduled_Time'])
df['actual_full'] = pd.to_datetime(df['Date'] + ' ' + df['Actual_Time'])
time_difference = df['actual_full'] - df['scheduled_full']
df['delayed'] = (time_difference > pd.Timedelta('20 minutes')).astype(int)

In [4]:
#Step 2
import numpy as np

hours = df['scheduled_full'].dt.hour.values
condition = [hours < 12,(hours >= 12) & (hours < 18),hours >= 18]
times = ['Morning', 'Afternoon', 'Evening']
df['TimeOfDay'] = np.select(condition, times, default='unknown')
print(df['TimeOfDay'].value_counts())

TimeOfDay
Afternoon    1098
Morning       699
Evening       404
Name: count, dtype: int64


In [5]:
#Step 3
df = df[df['Carrier'] != 'OH']
df = df.reset_index(drop=True)
df.shape
df['Carrier'].value_counts()

Carrier
DH    551
RU    408
US    404
DL    388
MQ    295
CO     94
UA     31
Name: count, dtype: int64

In [6]:
#Step 4
attributes = ['Carrier','Destination','Distance','Origin','Weather','DayOfWeek','DayOfMonth','TimeOfDay']
x_data = df[attributes].copy()
x_data['DayOfWeek'] = x_data['DayOfWeek'].astype(str)
categorical = ['Carrier','Destination','Origin','TimeOfDay','DayOfWeek']
x = pd.get_dummies(x_data,columns=categorical,drop_first=False)
y = df['delayed']

In [7]:
#Step 5: Holdout
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.30, random_state = 1)

model = MLPClassifier(random_state=1)
model.fit(x_train,y_train)

predicted_test = model.predict(x_test)
print('Confusion Matrix:', confusion_matrix(y_test,predicted_test))
print('Accuracy:', accuracy_score(y_test,predicted_test))
print('Precision(Class 1):', precision_score(y_test,predicted_test,pos_label=1))
print('Recall(Class 1):', recall_score(y_test, predicted_test, pos_label=1))
print('Full Classification Report:', classification_report(y_test,predicted_test))

Confusion Matrix: [[548   7]
 [ 87  10]]
Accuracy: 0.8558282208588958
Precision(Class 1): 0.5882352941176471
Recall(Class 1): 0.10309278350515463
Full Classification Report:               precision    recall  f1-score   support

           0       0.86      0.99      0.92       555
           1       0.59      0.10      0.18        97

    accuracy                           0.86       652
   macro avg       0.73      0.55      0.55       652
weighted avg       0.82      0.86      0.81       652



In [11]:
#Step 5: Cross-validation
from sklearn.model_selection import KFold, cross_val_score
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
cv_model = MLPClassifier(random_state=1)
cv_accuracy = cross_val_score(cv_model, x,y,cv=kfold,scoring='accuracy').mean()
cv_precision = cross_val_score(cv_model, x,y,cv=kfold,scoring='precision').mean()
cv_recall = cross_val_score(cv_model, x,y,cv=kfold,scoring='recall').mean()
print('CV Accuracy:',round(cv_accuracy,4))
print('CV Precision:',round(cv_precision,4))
print('CV Recall:',round(cv_recall,4))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


CV Accuracy: 0.8756
CV Precision: 0.5533
CV Recall: 0.0473
